# 📦 PHẦN 1: TIỀN XỬ LÝ & KẾT NỐI 5 BẢNG DỮ LIỆU

- **Người phụ trách:** Đinh Quốc Anh (Data Engineer & Data Analyst)
- **Mục tiêu:** Làm sạch 5 bảng dữ liệu, chuẩn hóa mã định danh, xử lý missing values, JOIN thành bảng tổng thể, tạo 13 Calculated Fields
- **Input:** `data/raw/*.csv` (5 files)
- **Output:** `data/processed/merged_spotify_data.csv`

---

### 📌 Checklist công việc
- [x] Nạp và kiểm tra cấu trúc 5 file dữ liệu
- [x] Data Quality Report cho từng bảng
- [x] Làm sạch Weekly Charts (trích `track_id`, parse `week`, xử lý `streams`)
- [x] Làm sạch Tracks Dataset (dedup, validate audio features)
- [x] Làm sạch Best-Selling Artists (parse doanh số, fix encoding)
- [x] JOIN 5 bảng (LEFT JOIN từ Weekly Charts)
- [x] Tạo 13 Calculated Fields (`is_trending`, `mood_category`, `covid_phase`, …)
- [x] Xuất `merged_spotify_data.csv` và sample

### 🏗️ Sơ đồ kiến trúc Pipeline

```
┌─────────────────────────────────────────────────────────────┐
│                  5 BẢNG DỮ LIỆU GỐC                       │
│  weekly-charts ─┬─(track_id)──→ tracks-dataset             │
│    (314K rows)  ├─(country,year)──→ gdp_data               │
│                 ├─(country_code)──→ country_codes           │
│                 └─(artist_name)──→ best-selling-artists     │
├─────────────────────────────────────────────────────────────┤
│  BƯỚC 1: Nạp → BƯỚC 2-4: Làm sạch → BƯỚC 5: JOIN          │
│  → BƯỚC 6: Calculated Fields → EXPORT                      │
├─────────────────────────────────────────────────────────────┤
│  OUTPUT: merged_spotify_data.csv (300K+ rows, 50+ cols)     │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ════════════════════════════════════════════════════
# 0. KHỞI TẠO MÔI TRƯỜNG & THƯ VIỆN
# ════════════════════════════════════════════════════
import os
import sys
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Thêm thư mục gốc dự án vào PATH để import module
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project Root: {PROJECT_ROOT}')
print(f'Python {sys.version}')
print('Môi trường làm việc đã sẵn sàng! ✅')

In [ ]:
# Import pipeline module
from src.data_loader import (
    load_raw_data, clean_weekly_charts, clean_tracks_dataset,
    clean_best_selling_artists, join_all_tables, create_calculated_fields,
    data_quality_report, AUDIO_FEATURES, AUDIO_0_1,
    RAW_DIR, PROCESSED_DIR, SAMPLE_DIR
)
print('Import src.data_loader thành công! ✅')

---
## 📂 BƯỚC 1: NẠP DỮ LIỆU THÔ

Dự án sử dụng **5 tập dữ liệu** từ nhiều nguồn khác nhau:

| # | File | Nguồn | Ý nghĩa |
|---|------|-------|--------|
| 1 | `spotify-weekly-charts.csv` | Spotify Weekly Charts | Bảng xếp hạng tuần kèm audio features (~835 MB) |
| 2 | `spotify-tracks-dataset.csv` | Spotify Web API / Kaggle | 114K bài hát với đặc trưng âm thanh |
| 3 | `gdp_data.csv` | World Bank | GDP quốc gia theo năm (1960–2022) |
| 4 | `country_codes.csv` | World Bank | Mã quốc gia, khu vực, nhóm thu nhập |
| 5 | `best-selling-artists.csv` | Wikipedia/RIAA | 121 nghệ sĩ bán đĩa nhiều nhất lịch sử |

> **Lưu ý:** File weekly-charts ~835 MB rất lớn. Trong notebook này, ta demo với **sample 25,000 dòng** từ `data/sample/spotify_sample_clean.csv` để chạy nhanh. Pipeline đầy đủ chạy bằng `python src/data_loader.py`.

In [ ]:
# ════════════════════════════════════════════════════
# Nạp 5 bảng dữ liệu thô
# ════════════════════════════════════════════════════

# Sử dụng sample data cho demo nhanh
USE_SAMPLE = True  # Đổi thành False để chạy trên full data

if USE_SAMPLE:
    sample_path = os.path.join(SAMPLE_DIR, 'spotify_sample_clean.csv')
    print(f'📂 Nạp sample data: {sample_path}')
    charts_raw = pd.read_csv(sample_path, low_memory=False)
    print(f'   Weekly Charts (sample): {charts_raw.shape[0]:,} dòng × {charts_raw.shape[1]} cột')
else:
    charts_raw = pd.read_csv(os.path.join(RAW_DIR, 'spotify-weekly-charts.csv'), low_memory=False)
    print(f'   Weekly Charts (full): {charts_raw.shape[0]:,} dòng × {charts_raw.shape[1]} cột')

tracks_raw = pd.read_csv(os.path.join(RAW_DIR, 'spotify-tracks-dataset.csv'), low_memory=False)
gdp_raw = pd.read_csv(os.path.join(RAW_DIR, 'gdp_data.csv'))
country_codes_raw = pd.read_csv(os.path.join(RAW_DIR, 'country_codes.csv'))
artists_raw = pd.read_csv(os.path.join(RAW_DIR, 'best-selling-artists.csv'))

print(f'   Tracks Dataset: {tracks_raw.shape[0]:,} dòng × {tracks_raw.shape[1]} cột')
print(f'   GDP Data: {gdp_raw.shape[0]:,} dòng × {gdp_raw.shape[1]} cột')
print(f'   Country Codes: {country_codes_raw.shape[0]:,} dòng × {country_codes_raw.shape[1]} cột')
print(f'   Best-Selling Artists: {artists_raw.shape[0]:,} dòng × {artists_raw.shape[1]} cột')

total = charts_raw.shape[0] + tracks_raw.shape[0] + gdp_raw.shape[0] + country_codes_raw.shape[0] + artists_raw.shape[0]
print(f'\n📊 Tổng cộng: {total:,} bản ghi từ 5 bảng')

---
## 📋 BƯỚC 1b: DATA QUALITY REPORT

Trước khi làm sạch, kiểm tra chất lượng dữ liệu của từng bảng:
- Kiểu dữ liệu mỗi cột
- Số lượng / Tỷ lệ giá trị thiếu (Missing Values)
- Số lượng giá trị duy nhất (Unique Values)
- Mẫu dữ liệu

In [ ]:
# ════════════════════════════════════════════════════
# Data Quality Report — Bảng 1: Weekly Charts
# ════════════════════════════════════════════════════
print('\n' + '='*70)
print('📋 BẢNG 1: SPOTIFY WEEKLY CHARTS')
print('='*70)
print(f'Shape: {charts_raw.shape}')
print(f'\nKiểu dữ liệu:')
print(charts_raw.dtypes)
print(f'\nMissing values:')
missing = charts_raw.isnull().sum()
missing_pct = (charts_raw.isnull().mean() * 100).round(2)
print(pd.DataFrame({'Missing': missing, '%': missing_pct}).query('Missing > 0'))
print(f'\n📌 Mẫu dữ liệu (3 dòng đầu):')
charts_raw.head(3)

In [ ]:
# ════════════════════════════════════════════════════
# Data Quality Report — Bảng 2: Tracks Dataset
# ════════════════════════════════════════════════════
print('\n' + '='*70)
print('📋 BẢNG 2: SPOTIFY TRACKS DATASET')
print('='*70)
print(f'Shape: {tracks_raw.shape}')
print(f'Track_id duy nhất: {tracks_raw["track_id"].nunique():,} / {len(tracks_raw):,} tổng')
print(f'\nMissing values:')
missing2 = tracks_raw.isnull().sum()
missing2_pct = (tracks_raw.isnull().mean() * 100).round(2)
print(pd.DataFrame({'Missing': missing2, '%': missing2_pct}).query('Missing > 0'))
print(f'\nThống kê Audio Features:')
audio_cols = ['danceability', 'energy', 'valence', 'tempo', 'loudness', 'acousticness']
available_audio = [c for c in audio_cols if c in tracks_raw.columns]
tracks_raw[available_audio].describe().round(3)

In [ ]:
# ════════════════════════════════════════════════════
# Data Quality Report — Bảng 3, 4, 5
# ════════════════════════════════════════════════════
print('\n' + '='*70)
print('📋 BẢNG 3: GDP DATA')
print('='*70)
print(f'Shape: {gdp_raw.shape}')
print(f'Quốc gia: {gdp_raw["country_name"].nunique()} | Năm: {gdp_raw["year"].min()}-{gdp_raw["year"].max()}')
print(f'GDP missing: {gdp_raw["value"].isnull().sum():,} ({gdp_raw["value"].isnull().mean()*100:.1f}%)')
print(f'\n' + '='*70)
print('📋 BẢNG 4: COUNTRY CODES')
print('='*70)
print(f'Shape: {country_codes_raw.shape}')
print(f'Regions: {country_codes_raw["region"].nunique()}')
print(f'Income groups: {country_codes_raw["income_group"].unique().tolist()}')
print(f'\n' + '='*70)
print('📋 BẢNG 5: BEST-SELLING ARTISTS')
print('='*70)
print(f'Shape: {artists_raw.shape}')
print(f'Cột: {artists_raw.columns.tolist()}')
print(f'\nMẫu dữ liệu:')
artists_raw[['Artist name', 'Country', 'Genre']].head(5)

---
## 🧹 BƯỚC 2: LÀM SẠCH BẢNG XẾP HẠNG TUẦN

Đây là bảng dữ liệu lõi (314,000+ dòng). Các vấn đề cần xử lý:

1. **Cột `uri`** chứa mã bài hát dạng `spotify:track:XXXX` → cần trích xuất `track_id`
2. **Cột `week`** dạng text → cần chuyển `datetime`, trích `year`, `month`, `quarter`
3. **Cột `streams`** có thể chứa NaN hoặc giá trị ≤ 0 → loại bỏ
4. **Trùng lặp** bản ghi (cùng track_id + week + country) → deduplicate
5. **Cột `region`** (Spotify) xung đột tên với `region` (World Bank) → đổi tên
6. **Cột dư thừa** (artist_img, album_cover, pivot) → loại bỏ

In [ ]:
# ════════════════════════════════════════════════════
# Áp dụng hàm clean_weekly_charts từ pipeline
# ════════════════════════════════════════════════════
charts_clean = clean_weekly_charts(charts_raw)

# Kiểm tra kết quả
print(f'\n📌 Kiểm tra sau khi làm sạch:')
print(f'   Shape: {charts_clean.shape}')
print(f'   Cột: {charts_clean.columns.tolist()}')
print(f'   track_id sample: {charts_clean["track_id"].iloc[0]}')
print(f'   week dtype: {charts_clean["week"].dtype}')
print(f'   year range: {charts_clean["year"].min()} - {charts_clean["year"].max()}')
print(f'   streams range: {charts_clean["streams"].min():,} - {charts_clean["streams"].max():,}')
charts_clean.head(3)

---
## 🧹 BƯỚC 3: LÀM SẠCH BẢNG ĐẶC TRƯNG BÀI HÁT

Bảng `spotify-tracks-dataset.csv` chứa 114,000 bài hát với đặc trưng âm thanh.

**Vấn đề cần xử lý:**
- Cột `Unnamed: 0` (index dư thừa)
- Trùng `track_id` → giữ bản ghi có `popularity` cao nhất
- Audio features ngoài phạm vi [0, 1]
- `duration_ms` bất thường (< 30s hoặc > 15 phút)

> **Lưu ý:** Audio features (danceability, energy, v.v.) đã có sẵn trong bảng Weekly Charts. Bảng Tracks chỉ bổ sung: `track_genre`, `popularity`, `duration_ms`, `explicit`.

In [ ]:
# ════════════════════════════════════════════════════
# Áp dụng hàm clean_tracks_dataset
# ════════════════════════════════════════════════════
tracks_clean = clean_tracks_dataset(tracks_raw)

print(f'\n📌 Kiểm tra:')
print(f'   Shape: {tracks_clean.shape}')
print(f'   Cột: {tracks_clean.columns.tolist()}')
print(f'   Genres duy nhất: {tracks_clean["track_genre"].nunique()}')
print(f'   Top 10 genres:')
print(tracks_clean['track_genre'].value_counts().head(10))

---
## 🧹 BƯỚC 4: LÀM SẠCH BẢNG NGHỆ SĨ BÁN ĐĨA

Bảng `best-selling-artists.csv` chứa 121 nghệ sĩ bán đĩa nhiều nhất mọi thời đại.

**Vấn đề cần xử lý:**
- Cột `Total certified units` là text phức tạp (ví dụ: `"294.6 millionUS: 217.250 million..."`) → parse ra số triệu
- Cột `Active years` bị lỗi encoding (`1960â€"1970`) → trích `career_start` và `career_end`
- Chuẩn hóa tên nghệ sĩ (lower-case) để JOIN với bảng Charts

In [ ]:
# ════════════════════════════════════════════════════
# Áp dụng hàm clean_best_selling_artists
# ════════════════════════════════════════════════════
artists_clean = clean_best_selling_artists(artists_raw)

print(f'\n📌 Kiểm tra:')
print(f'   Shape: {artists_clean.shape}')
print(f'   Cột: {artists_clean.columns.tolist()}')
print(f'\n   Top 10 nghệ sĩ bán đĩa nhiều nhất:')
top10 = artists_clean.nlargest(10, 'certified_units_million')
top10[['legacy_artist_name', 'certified_units_million', 'career_start', 'legacy_artist_genre']]

---
## 🔗 BƯỚC 5: JOIN 5 BẢNG DỮ LIỆU

Kết nối 5 bảng thành **một bảng tổng thể duy nhất** sử dụng **LEFT JOIN** từ Weekly Charts:

```
JOIN 1: charts ⋈ gdp_data        ON (country = country_name, year = year)  → GDP value
JOIN 2: result  ⋈ country_codes   ON (country_code)                        → region, income_group
JOIN 3: result  ⋈ tracks_dataset  ON (track_id)                            → genre, popularity
JOIN 4: result  ⋈ best_selling    ON (artist_name ≈ artist_individual)     → legacy status
```

> **Tại sao dùng LEFT JOIN?** Vì bảng Weekly Charts là bảng lõi — ta muốn giữ tất cả bản ghi từ charts, dù các bảng lookup (GDP, tracks) có thể không match 100%.

In [ ]:
# ════════════════════════════════════════════════════
# Áp dụng hàm join_all_tables
# ════════════════════════════════════════════════════
merged = join_all_tables(
    charts_clean, tracks_clean, gdp_raw, country_codes_raw, artists_clean
)

print(f'\n📌 Kiểm tra sau JOIN:')
print(f'   Shape: {merged.shape}')
print(f'   Quốc gia: {merged["country"].nunique()}')
print(f'   GDP match: {merged["gdp_value"].notna().mean()*100:.1f}%')
print(f'   Region match: {merged["region_wb"].notna().mean()*100:.1f}%')
print(f'   Income group match: {merged["income_group"].notna().mean()*100:.1f}%')
print(f'   Genre match: {merged["track_genre"].notna().mean()*100:.1f}%')
print(f'   Legacy artist match: {merged["legacy_artist_name"].notna().sum():,}')
print(f'\n   Cột ({len(merged.columns)}):')
print(f'   {merged.columns.tolist()}')

---
## 🧮 BƯỚC 6: TẠO CALCULATED FIELDS

Tạo **13 trường tính toán mới** phục vụ trực tiếp các hướng phân tích trong đồ án:

| # | Trường | Công thức | Phục vụ Insight |
|---|--------|-----------|----------------|
| 1 | `is_trending` | `rank ≤ 50` → 1 | ML: Logistic Regression dự báo Hit |
| 2 | `is_popular` | `popularity ≥ 70` → 1 | Phân tích bài phổ biến toàn cầu |
| 3 | `mood_category` | valence × energy quadrant | Bản đồ tâm trạng thế giới |
| 4 | `stream_share_country` | streams / tổng streams (country, week) | Thị phần nghe nhạc |
| 5 | `covid_phase` | 2020/2021/2022 → 3 giai đoạn | Phân tích COVID-19 |
| 6 | `energy_loudness_index` | (energy + norm(loudness)) / 2 | Chỉ số sôi động tổng hợp |
| 7 | `is_latin` | Detect từ genre + language | Theo dõi Latin Wave |
| 8 | `is_kpop` | Detect từ genre + language | Theo dõi K-Pop invasion |
| 9 | `is_legacy_artist` | Có trong best-selling → 1 | Sức sống huyền thoại |
| 10 | `duration_min` | duration / 60000 | Thời lượng dễ đọc |
| 11 | `audio_diversity_score` | std(danceability, energy, valence, acousticness) | Đa dạng gu nhạc |
| 12 | `log_streams` | log(1 + streams) | ML & tương quan |
| 13 | `gdp_log` | log(1 + gdp_value) | Scatter plot GDP |

In [ ]:
# ════════════════════════════════════════════════════
# Áp dụng hàm create_calculated_fields
# ════════════════════════════════════════════════════
final_df = create_calculated_fields(merged)

print(f'\n📌 Kiểm tra Calculated Fields:')
print(f'   Shape: {final_df.shape}')

# Validate từng field
calc_fields = ['is_trending', 'is_popular', 'mood_category', 'stream_share_country',
               'covid_phase', 'energy_loudness_index', 'is_latin', 'is_kpop',
               'is_legacy_artist', 'duration_min', 'audio_diversity_score',
               'log_streams', 'gdp_log']

for field in calc_fields:
    if field in final_df.columns:
        print(f'   ✅ {field}: dtype={final_df[field].dtype}, '
              f'null={final_df[field].isnull().sum()}, '
              f'unique={final_df[field].nunique()}')
    else:
        print(f'   ❌ {field}: KHÔNG TÌM THẤY')

In [ ]:
# ════════════════════════════════════════════════════
# Kiểm tra phân phối Mood Category
# ════════════════════════════════════════════════════
print('\n📊 Phân phối Mood Category:')
mood_dist = final_df['mood_category'].value_counts()
mood_pct = final_df['mood_category'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': mood_dist, '%': mood_pct.round(1)}))

print('\n📊 Phân phối COVID Phase:')
covid_dist = final_df['covid_phase'].value_counts()
covid_pct = final_df['covid_phase'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': covid_dist, '%': covid_pct.round(1)}))

print('\n📊 Phân phối Latin vs K-Pop:')
print(f'   Latin tracks: {final_df["is_latin"].sum():,} ({final_df["is_latin"].mean()*100:.1f}%)')
print(f'   K-Pop tracks: {final_df["is_kpop"].sum():,} ({final_df["is_kpop"].mean()*100:.1f}%)')
print(f'   Legacy artist tracks: {final_df["is_legacy_artist"].sum():,} ({final_df["is_legacy_artist"].mean()*100:.1f}%)')

---
## 📊 BƯỚC 7: DATA QUALITY CHECK — BỘ DỮ LIỆU CUỐI CÙNG

Kiểm tra toàn diện bảng dữ liệu tổng hợp trước khi export.

In [ ]:
# ════════════════════════════════════════════════════
# Báo cáo chất lượng cuối cùng
# ════════════════════════════════════════════════════
print('='*70)
print('📊 FINAL DATA QUALITY REPORT')
print('='*70)
print(f'Tổng dòng: {len(final_df):,}')
print(f'Tổng cột: {len(final_df.columns)}')

# Tỷ lệ missing tổng
total_cells = len(final_df) * len(final_df.columns)
total_null = final_df.isnull().sum().sum()
print(f'Tổng cells: {total_cells:,}')
print(f'Missing cells: {total_null:,} ({total_null/total_cells*100:.2f}%)')

# Chi tiết từng cột
print(f'\nChi tiết từng cột:')
quality = pd.DataFrame({
    'dtype': final_df.dtypes.astype(str),
    'non_null': final_df.notna().sum(),
    'null': final_df.isnull().sum(),
    'null_pct': (final_df.isnull().mean() * 100).round(2),
    'unique': final_df.nunique()
})
quality

In [ ]:
# ════════════════════════════════════════════════════
# Thống kê mô tả các cột số (Audio Features + Streams)
# ════════════════════════════════════════════════════
num_cols = ['danceability', 'energy', 'valence', 'tempo', 'loudness',
            'acousticness', 'speechiness', 'instrumentalness', 'liveness',
            'streams', 'rank', 'weeks_on_chart']
available = [c for c in num_cols if c in final_df.columns]

print('📊 Thống kê mô tả các biến số:')
final_df[available].describe().round(3)

In [ ]:
# ════════════════════════════════════════════════════
# Thống kê theo quốc gia & nhóm thu nhập
# ════════════════════════════════════════════════════
print('📊 Số lượng bản ghi theo Quốc gia (Top 15):')
print(final_df['country'].value_counts().head(15))

print(f'\n📊 Phân phối theo Nhóm thu nhập (Income Group):')
if 'income_group' in final_df.columns:
    print(final_df['income_group'].value_counts())

print(f'\n📊 Phân phối theo Khu vực World Bank (region_wb):')
if 'region_wb' in final_df.columns:
    print(final_df['region_wb'].value_counts())

---
## 💾 BƯỚC 8: EXPORT DỮ LIỆU

Xuất bảng dữ liệu tổng hợp thành file CSV:
- **Full dataset:** `data/processed/merged_spotify_data.csv`
- **Sample (20K dòng):** `data/sample/spotify_sample_merged.csv`

In [ ]:
# ════════════════════════════════════════════════════
# Export dữ liệu
# ════════════════════════════════════════════════════
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Lưu full dataset
full_path = os.path.join(PROCESSED_DIR, 'merged_spotify_data.csv')
final_df.to_csv(full_path, index=False)
mb = os.path.getsize(full_path) / 1024 / 1024
print(f'✅ Full dataset saved: {full_path}')
print(f'   {len(final_df):,} dòng × {len(final_df.columns)} cột ({mb:.1f} MB)')

# Lưu sample (20K dòng)
sample_df = final_df.sample(n=min(20000, len(final_df)), random_state=42)
sample_path = os.path.join(SAMPLE_DIR, 'spotify_sample_merged.csv')
sample_df.to_csv(sample_path, index=False)
smb = os.path.getsize(sample_path) / 1024 / 1024
print(f'\n✅ Sample saved: {sample_path}')
print(f'   {len(sample_df):,} dòng × {len(sample_df.columns)} cột ({smb:.1f} MB)')

---
## 📊 QUICK PREVIEW — Các insight ban đầu từ dữ liệu

Trước khi chuyển sang EDA chuyên sâu (`02_eda.ipynb`), hãy kiểm tra nhanh một số insight dự kiến.

In [ ]:
# ════════════════════════════════════════════════════
# Quick Insight 1: GDP vs Streams correlation
# ════════════════════════════════════════════════════
if 'gdp_value' in final_df.columns:
    # Tương quan GDP vs Streams theo quốc gia
    country_agg = final_df.groupby('country').agg({
        'gdp_value': 'mean',
        'streams': 'sum',
        'valence': 'mean',
        'danceability': 'mean',
        'energy': 'mean'
    }).dropna()
    
    if len(country_agg) > 5:
        corr_streams = country_agg['gdp_value'].corr(country_agg['streams'])
        corr_valence = country_agg['gdp_value'].corr(country_agg['valence'])
        corr_dance = country_agg['gdp_value'].corr(country_agg['danceability'])
        
        print('📊 Tương quan GDP với các chỉ số (theo quốc gia):')
        print(f'   GDP vs Total Streams : r = {corr_streams:.3f}')
        print(f'   GDP vs Valence       : r = {corr_valence:.3f}')
        print(f'   GDP vs Danceability  : r = {corr_dance:.3f}')
        
        print(f'\n💡 Insight: GDP ↔ Streams tương quan {"DƯƠNG MẠNH" if corr_streams > 0.5 else "yếu"}.')
        if corr_valence < 0:
            print(f'   → GDP ↔ Valence tương quan ÂM (r={corr_valence:.3f}): '
                  f'Nghịch lý Cảm xúc — nước giàu nghe nhạc BUỒN hơn!')
    else:
        print('⚠️ Không đủ quốc gia có GDP data để tính tương quan')

In [ ]:
# ════════════════════════════════════════════════════
# Quick Insight 2: Audio profile trung bình bài hit
# ════════════════════════════════════════════════════
hit_profile = final_df[final_df['is_trending'] == 1][[
    'danceability', 'energy', 'valence', 'tempo', 'loudness', 'acousticness'
]].mean()
non_hit = final_df[final_df['is_trending'] == 0][[
    'danceability', 'energy', 'valence', 'tempo', 'loudness', 'acousticness'
]].mean()

print('📊 "Công thức vàng" bài hát Hit (is_trending=1) vs Non-Hit:')
comparison = pd.DataFrame({'Hit (Top 50)': hit_profile, 'Non-Hit': non_hit})
comparison['Chênh lệch'] = comparison['Hit (Top 50)'] - comparison['Non-Hit']
comparison.round(3)

In [ ]:
# ════════════════════════════════════════════════════
# Quick Insight 3: Top nghệ sĩ theo tổng streams
# ════════════════════════════════════════════════════
top_artists = (final_df.groupby('artist_individual')['streams']
               .sum()
               .sort_values(ascending=False)
               .head(15))

print('📊 Top 15 nghệ sĩ theo tổng lượt stream:')
for i, (artist, streams) in enumerate(top_artists.items(), 1):
    print(f'   {i:2d}. {artist:<30s} {streams:>15,} streams')

In [ ]:
# ════════════════════════════════════════════════════
# Quick Insight 4: Gu âm nhạc theo Income Group
# ════════════════════════════════════════════════════
if 'income_group' in final_df.columns:
    income_audio = final_df.groupby('income_group')[[
        'danceability', 'energy', 'valence', 'acousticness', 'tempo'
    ]].mean()
    
    print('📊 Trung bình Audio Features theo Nhóm thu nhập:')
    print(income_audio.round(3))
    print()
    
    # Kiểm tra "Nghịch lý Cảm xúc"
    if 'High income' in income_audio.index:
        hi = income_audio.loc['High income', 'valence']
        labels_mid = [l for l in income_audio.index if 'middle' in l.lower()]
        if labels_mid:
            mid = income_audio.loc[labels_mid, 'valence'].mean()
            print(f'💡 Valence: High income = {hi:.3f} vs Middle income = {mid:.3f}')
            if hi < mid:
                print('   → XÁC NHẬN Nghịch lý Cảm xúc: Nước giàu nghe nhạc ÍT VUI hơn!')

---
## ✅ TỔNG KẾT

Pipeline 6 bước đã hoàn thành:

| Bước | Mô tả | Kết quả |
|------|-------|---------|
| 1 | Nạp dữ liệu thô | 5 bảng CSV nạp thành công |
| 2 | Làm sạch Weekly Charts | Trích `track_id`, parse `week`, xử lý `streams` |
| 3 | Làm sạch Tracks Dataset | Dedup `track_id`, validate audio features |
| 4 | Làm sạch Best-Selling Artists | Parse doanh số, fix encoding |
| 5 | JOIN 5 bảng | LEFT JOIN tạo bảng tổng thể |
| 6 | Calculated Fields | 13 trường tính toán mới |

**Output files:**
- `data/processed/merged_spotify_data.csv` — Dataset đầy đủ
- `data/sample/spotify_sample_merged.csv` — Sample 20K dòng

**Bước tiếp theo:** Phân tích khám phá dữ liệu đa chiều (EDA) trong `02_eda.ipynb`

> 💡 Để chạy pipeline trên **toàn bộ dữ liệu gốc** (314K+ dòng), chạy lệnh:
> ```bash
> python src/data_loader.py
> ```
> Hoặc dùng chế độ sample:
> ```bash
> python src/data_loader.py --sample 50000
> ```